Install Dependencies

In [ ]:
!pip install datasets
!pip install transformers -U
!pip install accelerate -U
!pip install trl
!pip install bitsandbytes
!pip install peft

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Load Dataset

In [ ]:
from datasets import load_dataset
DATASET_NAME = "ChrisHayduk/Llama-2-SQL-Dataset"
dataset = load_dataset(DATASET_NAME)

In [ ]:
full_training_dataset = dataset["train"]
shuffled = full_training_dataset.shuffle()
training_dataset = shuffled.select(range(5000)) # Picking only 5000 random datapoints as that's usually enough for finetuning

In [ ]:
len(shuffled)

Quantization

In [ ]:
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16"
)

Importing llama-2-7b and its corresponding tokenizer

In [ ]:
import transformers
from transformers import AutoModelForCausalLM
from transformers import AutoTokenizer

MODEL_NAME = "NousResearch/Llama-2-7b-hf"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto"
)

model.config.use_cache = True

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

Prepare Data

In [ ]:
def construct_datapoint(x):
  combined = x['input'] + x['output'] + tokenizer.eos_token
  return tokenizer(combined, padding=True)
training_dataset = training_dataset.map(construct_datapoint)

In [ ]:
print(training_dataset)

Configure LoRA

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj','k_proj','down_proj','v_proj','gate_proj','o_proj','up_proj'],
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = prepare_model_for_kbit_training(model)
model = get_peft_model(model,peft_config)

Set Generation Configuration Parameters

In [ ]:
generation_configuration = model.generation_config
generation_configuration.pad_token_id = tokenizer.eos_token_id
generation_configuration.eos_token_id = tokenizer.eos_token_id
generation_configuration.max_new_tokens = 80
generation_configuration.temperature = 0.7
generation_configuration.top_p = 0.9
generation_configuration.top_k = 50
generation_configuration.do_sample = True

Training

In [ ]:
train_arguments = transformers.TrainingArguments(
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=3e-5,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    output_dir="fine_tuning"
)

trainer = transformers.Trainer(
    model=model,
    train_dataset=training_dataset,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer,mlm=False),
    args=train_arguments
)

model.config.use_cache = False

In [ ]:
trainer.train()

Function to Generate Response

In [ ]:
import torch

def generate_sql(question: str):
    model.eval()
    inputs = tokenizer(question, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    # pick a reliable EOS id
    eos_id = tokenizer.eos_token_id
    if eos_id is None:
        eos_id = tokenizer.convert_tokens_to_ids("</s>")  # LLaMA-style fallback

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False,               # greedy; temperature is ignored here
            eos_token_id=eos_id,
            pad_token_id=eos_id,
        )

    # only the completion
    gen_ids = out[0, input_len:]

    # HARD STOP: truncate at first EOS token id (and exclude it)
    eos_pos = (gen_ids == eos_id).nonzero(as_tuple=True)[0]
    if eos_pos.numel() > 0:
        gen_ids = gen_ids[: eos_pos[0].item()]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    text = text.split("</s>", 1)[0]
    return text


Trying Out The Model

In [ ]:
evaluation_dataset=dataset['eval'].shuffle()

sample_sql_question = evaluation_dataset[0]['input']
correct_answer = evaluation_dataset[0]['output']
generate_sql(sample_sql_question)

In [ ]:
len(evaluation_dataset)

In [ ]:
print(correct_answer)

In [ ]:
generate_sql(sample_sql_question) == correct_answer

Save Model

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, shutil

save_dir = "/content/drive/MyDrive/llama_sql_finetune"

# nuke the folder so no stale files remain
if os.path.exists(save_dir):
    shutil.rmtree(save_dir)

os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir, safe_serialization=True)
tokenizer.save_pretrained(save_dir)

print("Saved cleanly to:", save_dir)

Load Model

In [6]:
from google.colab import drive
drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/llama_sql_finetune"

Mounted at /content/drive


In [7]:

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(save_dir)

model = AutoModelForCausalLM.from_pretrained(
    save_dir,
    torch_dtype=torch.float16,   # or bfloat16 if you trained in bf16
    device_map="auto",
)

model.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): lora.Linear(
            (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
              (default): Dropout(p=0.05, inplace=False)
            )
            (lora_A): ModuleDict(
              (default): Linear(in_features=4096, out_features=16, bias=False)
            )
            (lora_B): ModuleDict(
              (default): Linear(in_features=16, out_features=4096, bias=False)
            )
            (lora_embedding_A): ParameterDict()
            (lora_embedding_B): ParameterDict()
            (lora_magnitude_vector): ModuleDict()
          )
          (k_proj): lora.Linear(
            (base_layer): Linear(in_features=4096, out_features=4096, bias=False)
            (lora_dropout): ModuleDict(
        

App

In [8]:
!pip -q install streamlit transformers accelerate torch


In [9]:
import os, textwrap

os.makedirs(os.path.expanduser("~/.streamlit"), exist_ok=True)

config = textwrap.dedent("""\
[server]
headless = true
enableCORS = false
enableXsrfProtection = false
port = 8501
address = "0.0.0.0"
""")

with open(os.path.expanduser("~/.streamlit/config.toml"), "w") as f:
    f.write(config)

print("Wrote:", os.path.expanduser("~/.streamlit/config.toml"))


Wrote: /root/.streamlit/config.toml


In [10]:
%%writefile app.py
# app.py
# Run inside Colab:
#   !pip -q install streamlit transformers accelerate torch
#   !streamlit run app.py --server.port 8501 --server.address 0.0.0.0
#
# Then expose with cloudflared/ngrok (as shown earlier).

import re
import time
from pathlib import Path

import torch
import streamlit as st
from transformers import AutoTokenizer, AutoModelForCausalLM


# -----------------------------
# Prompt template (your required hidden text)
# -----------------------------
HIDDEN_PREAMBLE = (
    "Below is an instruction that describes a SQL generation task, paired with an input "
    "that provides further context about the available table schemas. Write SQL code that "
    "appropriately answers the request.\n"
)

def build_prompt(instruction: str, input_schema: str) -> str:
    return (
        f"{HIDDEN_PREAMBLE}\n"
        f"### Instruction:\n{instruction.strip()}\n\n"
        f"### Input:\n{input_schema.strip()}\n\n"
        f"### Response:\n"
    )


# -----------------------------
# Post-processing / stopping
# -----------------------------
def truncate_on_eos(gen_ids: torch.Tensor, eos_id: int) -> torch.Tensor:
    """Truncate token ids BEFORE first EOS token (exclude eos)."""
    if eos_id is None:
        return gen_ids
    eos_pos = (gen_ids == eos_id).nonzero(as_tuple=True)[0]
    if eos_pos.numel() > 0:
        return gen_ids[: eos_pos[0].item()]
    return gen_ids

def postprocess_sql(text: str) -> str:
    text = text.strip()

    # Remove code fences if any
    text = re.sub(r"^\s*```(?:sql)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```\s*$", "", text).strip()

    # Strip literal eos text if it appears
    text = text.split("</s>", 1)[0].strip()

    # Stop if it starts emitting the next example / headers
    text = re.split(r"\n\s*(?:###\s*Instruction:|Below is an instruction)", text, maxsplit=1)[0].strip()

    return text


# -----------------------------
# Load model/tokenizer (cached)
# -----------------------------
@st.cache_resource(show_spinner=False)
def load_local_model(model_dir: str, dtype_str: str = "float16"):
    model_path = Path(model_dir)
    if not model_path.exists():
        raise FileNotFoundError(f"Model folder not found: {model_path.resolve()}")

    dtype_map = {
        "float16": torch.float16,
        "bfloat16": torch.bfloat16,
        "float32": torch.float32,
    }
    torch_dtype = dtype_map.get(dtype_str, torch.float16)

    tokenizer = AutoTokenizer.from_pretrained(str(model_path), use_fast=True)

    # LLaMA often has no pad token: set pad=eos
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        str(model_path),
        torch_dtype=torch_dtype,
        device_map="auto",
    )
    model.eval()
    return model, tokenizer


@torch.no_grad()
def generate_sql(model, tokenizer, prompt: str,
                 max_new_tokens: int = 128,
                 do_sample: bool = False,
                 temperature: float = 0.8,
                 top_k: int = 0,
                 top_p: float = 1.0):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    eos_id = tokenizer.eos_token_id
    if eos_id is None:
        eos_id = tokenizer.convert_tokens_to_ids("</s>")

    gen_kwargs = dict(
        **inputs,
        max_new_tokens=int(max_new_tokens),
        do_sample=bool(do_sample),
        eos_token_id=eos_id,
        pad_token_id=tokenizer.pad_token_id,
    )

    if do_sample:
        gen_kwargs["temperature"] = float(temperature)
        if int(top_k) > 0:
            gen_kwargs["top_k"] = int(top_k)
        if 0 < float(top_p) <= 1.0:
            gen_kwargs["top_p"] = float(top_p)

    out = model.generate(**gen_kwargs)

    gen_ids = out[0][input_len:]
    gen_ids = truncate_on_eos(gen_ids, eos_id)

    text = tokenizer.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return postprocess_sql(text)


# -----------------------------
# Streamlit UI
# -----------------------------
st.set_page_config(page_title="SQL Generator", page_icon="🧾", layout="wide")
st.title("🧾 SQL Query Generator (Fine-tuned LLaMA)")

with st.sidebar:
    st.header("Model")
    model_dir = st.text_input(
        "Model folder",
        value="/content/drive/MyDrive/llama_sql_finetune",
        help="This should contain config.json + model weights + tokenizer files."
    )
    dtype = st.selectbox("dtype", ["float16", "bfloat16", "float32"], index=0)

    st.divider()
    st.header("Generation")
    max_new_tokens = st.slider("max_new_tokens", 16, 512, 128, 8)
    do_sample = st.checkbox("do_sample (sampling)", value=False)
    temperature = st.slider("temperature", 0.1, 2.0, 0.8, 0.1)
    top_k = st.number_input("top_k (0 disables)", min_value=0, max_value=500, value=0, step=10)
    top_p = st.slider("top_p", 0.1, 1.0, 1.0, 0.05)

    st.divider()
    show_prompt = st.checkbox("Show prompt", value=False)

# Inputs
col1, col2 = st.columns(2, gap="large")
with col1:
    instruction = st.text_area("### Instruction", height=180, placeholder="e.g. What is the result on November 1, 1992?")
with col2:
    input_schema = st.text_area("### Input", height=180, placeholder="e.g. CREATE TABLE table_name_53 (result VARCHAR, date VARCHAR)")

# Buttons
b1, b2, b3 = st.columns([1, 1, 2])
with b1:
    load_btn = st.button("🔌 Load model", type="primary")
with b2:
    gen_btn = st.button("⚡ Generate")
with b3:
    st.caption("Runs entirely in Colab. Use a tunnel (cloudflared/ngrok) to open in your browser.")

if "model_loaded" not in st.session_state:
    st.session_state.model_loaded = False

if load_btn:
    with st.spinner("Loading model..."):
        try:
            model, tokenizer = load_local_model(model_dir, dtype)
            st.session_state.model = model
            st.session_state.tokenizer = tokenizer
            st.session_state.model_loaded = True
            st.success("Model loaded.")
        except Exception as e:
            st.session_state.model_loaded = False
            st.error(str(e))

if gen_btn:
    if not instruction.strip() or not input_schema.strip():
        st.warning("Please fill both ### Instruction and ### Input.")
        st.stop()

    if not st.session_state.model_loaded:
        with st.spinner("Loading model..."):
            model, tokenizer = load_local_model(model_dir, dtype)
            st.session_state.model = model
            st.session_state.tokenizer = tokenizer
            st.session_state.model_loaded = True

    prompt = build_prompt(instruction, input_schema)

    if show_prompt:
        st.markdown("**Prompt sent to model:**")
        st.code(prompt, language="text")

    with st.spinner("Generating..."):
        t0 = time.time()
        sql = generate_sql(
            st.session_state.model,
            st.session_state.tokenizer,
            prompt=prompt,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )
        dt = time.time() - t0

    st.markdown("### ✅ Response")
    st.code(sql, language="sql")
    st.caption(f"Generated in {dt:.2f}s")


Overwriting app.py


In [11]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 > streamlit.log 2>&1 &


In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://127.0.0.1:8501

# Click the "https://....trycloudflare.com" link in the output to access the streamlit application

2026-01-27T06:28:26Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-01-27T06:28:26Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-01-27T06:28:31Z INF +--------------------------------------------------------------------------------------------+
2026-01-27T06:28:31Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-01-27T06:28:31Z INF |  https://allow-subscriptions-watts-change.trycloudflar

Uploading on Huggingface (Ignore all the code below this for now)

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")



In [ ]:
# save_dir = "/content/drive/MyDrive/llama_sql_finetune"

In [ ]:
# !pip install transformers

In [ ]:
# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# tokenizer = AutoTokenizer.from_pretrained(save_dir)

# model = AutoModelForCausalLM.from_pretrained(
#     save_dir,
#     dtype=torch.float16,   # or bfloat16 if you trained in bf16
#     device_map="auto",
# )

In [ ]:
# !pip -q install --upgrade --force-reinstall "huggingface-hub>=0.34.0,<1.0"

In [ ]:
# from huggingface_hub import login
# login(token="HF_TOKEN") # replace it with your huggingface token


In [ ]:
# from huggingface_hub import HfApi, upload_folder

# HF_USERNAME = "srikara202"  # <- change this
# MODEL_REPO  = f"{HF_USERNAME}/llama-sql-finetune-model"

# api = HfApi()
# api.create_repo(repo_id=MODEL_REPO, repo_type="model", private=True, exist_ok=True)

# upload_folder(
#     repo_id=MODEL_REPO,
#     folder_path=save_dir,
#     repo_type="model",
# )
# print("Uploaded model to:", MODEL_REPO)


In [ ]:
# SPACE_REPO = f"{HF_USERNAME}/llama-sql-streamlit"


In [ ]:
# import os, textwrap
# os.makedirs("space_src", exist_ok=True)

# # 1) app.py (loads your model from the Hub repo)
# app_py = r'''
# import os
# import re
# import time
# from pathlib import Path

# import torch
# import streamlit as st
# from transformers import AutoTokenizer, AutoModelForCausalLM

# HIDDEN_PREAMBLE = (
#     "Below is an instruction that describes a SQL generation task, paired with an input "
#     "that provides further context about the available table schemas. Write SQL code that "
#     "appropriately answers the request.\n"
# )

# def build_prompt(instruction: str, input_schema: str) -> str:
#     return (
#         f"{HIDDEN_PREAMBLE}\n"
#         f"### Instruction:\n{instruction.strip()}\n\n"
#         f"### Input:\n{input_schema.strip()}\n\n"
#         f"### Response:\n"
#     )

# def truncate_on_eos(gen_ids: torch.Tensor, eos_id: int) -> torch.Tensor:
#     if eos_id is None:
#         return gen_ids
#     eos_pos = (gen_ids == eos_id).nonzero(as_tuple=True)[0]
#     if eos_pos.numel() > 0:
#         return gen_ids[:eos_pos[0].item()]
#     return gen_ids

# def postprocess_sql(text: str) -> str:
#     text = text.strip()
#     text = re.sub(r"^\s*```(?:sql)?\s*", "", text, flags=re.IGNORECASE)
#     text = re.sub(r"\s*```\s*$", "", text).strip()
#     text = text.split("</s>", 1)[0].strip()
#     text = re.split(r"\n\s*(?:###\s*Instruction:|Below is an instruction)", text, maxsplit=1)[0].strip()
#     return text

# @st.cache_resource(show_spinner=False)
# def load_model_and_tokenizer(model_id: str, dtype: str):
#     dtype_map = {"float16": torch.float16, "bfloat16": torch.bfloat16, "float32": torch.float32}
#     torch_dtype = dtype_map.get(dtype, torch.float16)

#     # HF_TOKEN secret (if set in Space settings) will be used automatically by huggingface_hub. :contentReference[oaicite:5]{index=5}
#     tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
#     if tokenizer.pad_token_id is None:
#         tokenizer.pad_token = tokenizer.eos_token

#     model = AutoModelForCausalLM.from_pretrained(
#         model_id,
#         torch_dtype=torch_dtype,
#         device_map="auto",
#     )
#     model.eval()
#     return model, tokenizer

# @torch.no_grad()
# def generate_sql(model, tokenizer, prompt: str, max_new_tokens: int,
#                  do_sample: bool, temperature: float, top_k: int, top_p: float):
#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
#     input_len = inputs["input_ids"].shape[-1]
#     eos_id = tokenizer.eos_token_id

#     gen_kwargs = dict(
#         **inputs,
#         max_new_tokens=int(max_new_tokens),
#         do_sample=bool(do_sample),
#         eos_token_id=eos_id,
#         pad_token_id=tokenizer.pad_token_id,
#     )
#     if do_sample:
#         gen_kwargs["temperature"] = float(temperature)
#         if int(top_k) > 0:
#             gen_kwargs["top_k"] = int(top_k)
#         if 0 < float(top_p) <= 1.0:
#             gen_kwargs["top_p"] = float(top_p)

#     out = model.generate(**gen_kwargs)
#     gen_ids = out[0][input_len:]
#     gen_ids = truncate_on_eos(gen_ids, eos_id)
#     text = tokenizer.decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
#     return postprocess_sql(text)

# st.set_page_config(page_title="SQL Generator", page_icon="🧾", layout="wide")
# st.title("🧾 SQL Query Generator")

# with st.sidebar:
#     st.header("Model")
#     model_id = st.text_input("Model repo id", value=os.environ.get("MODEL_ID", "YOUR_USERNAME/llama-sql-finetune-model"))
#     dtype = st.selectbox("dtype", ["float16", "bfloat16", "float32"], index=0)

#     st.divider()
#     st.header("Generation")
#     max_new_tokens = st.slider("max_new_tokens", 16, 512, 128, 8)
#     do_sample = st.checkbox("do_sample", value=False)
#     temperature = st.slider("temperature", 0.1, 2.0, 0.8, 0.1)
#     top_k = st.number_input("top_k (0 disables)", min_value=0, max_value=500, value=0, step=10)
#     top_p = st.slider("top_p", 0.1, 1.0, 1.0, 0.05)

#     show_prompt = st.checkbox("Show prompt", value=False)

# col1, col2 = st.columns(2)
# with col1:
#     instruction = st.text_area("### Instruction", height=180)
# with col2:
#     input_schema = st.text_area("### Input", height=180)

# gen_btn = st.button("⚡ Generate", type="primary")

# if gen_btn:
#     if not instruction.strip() or not input_schema.strip():
#         st.warning("Please fill both ### Instruction and ### Input.")
#         st.stop()

#     with st.spinner("Loading model..."):
#         model, tokenizer = load_model_and_tokenizer(model_id, dtype)

#     prompt = build_prompt(instruction, input_schema)
#     if show_prompt:
#         st.code(prompt, language="text")

#     with st.spinner("Generating..."):
#         t0 = time.time()
#         sql = generate_sql(model, tokenizer, prompt, max_new_tokens, do_sample, temperature, top_k, top_p)
#         st.code(sql, language="sql")
#         st.caption(f"Done in {time.time()-t0:.2f}s")
# '''
# open("space_src/app.py", "w").write(app_py)

# # 2) requirements.txt
# req = """\
# streamlit
# torch
# transformers
# accelerate
# safetensors
# sentencepiece
# """
# open("space_src/requirements.txt", "w").write(req)

# # 3) README.md (Space YAML config) :contentReference[oaicite:6]{index=6}
# readme = """\
# ---
# title: LLaMA SQL Generator
# emoji: 🧾
# sdk: streamlit
# python_version: 3.10
# ---

# Streamlit app for SQL generation.
# """
# open("space_src/README.md", "w").write(readme)

# print("Wrote Space files to ./space_src")


In [ ]:
# from huggingface_hub import upload_folder

# SPACE_REPO = f"{HF_USERNAME}/llama-sql-streamlit"  # <- same as the Space you created in UI

# upload_folder(
#     repo_id=SPACE_REPO,
#     folder_path="space_src",
#     repo_type="space",
# )
# print("Uploaded Space code to:", SPACE_REPO)


In [ ]:
# from huggingface_hub import upload_file

# MODEL_ID = "srikara202/llama-sql-finetune-model"
# SAVE_DIR = "/content/drive/MyDrive/llama_sql_finetune"

# upload_file(
#     path_or_fileobj=f"{SAVE_DIR}/tokenizer.model",
#     path_in_repo="tokenizer.model",
#     repo_id=MODEL_ID,
#     repo_type="model",
# )